# 🚦 Traffic Accident Severity — Synthetic Dataset Generation
> **Output :** `traffic_accident_dataset.csv`  
> **Rows :** ~8 400 (8 000 base + 5 % duplicate rows)  
> **Columns :** 26 (21 signal + 4 noise + 1 target)  
> **Target :** `accident_severity` → Minor | Moderate | Severe  

---
### What this notebook does
Generates a realistic synthetic dataset with **intentional data quality issues** built in,  
so the preprocessing notebook has something meaningful to clean:

| Issue | Amount | Why |
|-------|--------|-----|
| Missing values | 10 % per key column | Practise null imputation |
| Duplicate rows | 5 % (~400 rows) | Practise deduplication |
| Categorical inconsistencies | ~15 % of weather & state | Practise standardisation |
| Gaussian noise on numerics | σ = 5–12 per feature | Makes patterns less trivially learnable |
| 4 noise / redundant columns | Constant or random | Practise feature selection |

---
## 📦 Cell 1 — Imports & Seeds

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Fix seeds for reproducibility — same CSV every run
np.random.seed(42)
random.seed(42)

N = 8000   # base row count before duplicates
print(f'Generating {N} base rows ...')

---
## 🗂️ Cell 2 — Schema Constants & Inconsistency Maps
> Defines the canonical category lists and the variant spellings that will be  
> **intentionally injected** to simulate real-world dirty data.

In [ ]:
SEVERITY_LABELS = {0: 'Minor', 1: 'Moderate', 2: 'Severe'}

# ── Canonical category lists ──────────────────────────────────────────────
weather_map   = ['Clear', 'Rainy', 'Foggy', 'Snowy', 'Windy', 'Stormy']
road_map      = ['Dry', 'Wet', 'Icy', 'Under Construction', 'Potholed']
vehicle_map   = ['Car', 'Truck', 'Motorcycle', 'Bus', 'SUV', 'Van']
light_map     = ['Daylight', 'Dusk', 'Dawn', 'Night - Lit', 'Night - Unlit']
junction_map  = ['None', 'T-Junction', 'Roundabout', 'Crossroads', 'Slip Road']
road_type_map = ['Highway', 'Urban Road', 'Rural Road', 'Expressway', 'One-way']

# ── Categorical inconsistency pools ──────────────────────────────────────
# WHY: Real datasets have the same value entered in multiple ways.
#      We inject ~15 % variant spellings to simulate this.
weather_inconsistent = {
    'Clear' : ['Clear', 'CLEAR', 'clear', 'Sunny', 'Fair'],
    'Rainy' : ['Rainy', 'Rain', 'rainy', 'Raining', 'Wet Weather'],
    'Foggy' : ['Foggy', 'Fog', 'foggy', 'Misty'],
    'Snowy' : ['Snowy', 'Snow', 'snowy', 'Snow Fall'],
    'Windy' : ['Windy', 'windy', 'High Wind'],
    'Stormy': ['Stormy', 'Storm', 'Thunderstorm'],
}

# US state inconsistencies: abbreviated vs full name vs lowercase
state_pool = [
    'NY', 'New York', 'N.Y.', 'ny',
    'CA', 'California', 'CA.', 'ca',
    'TX', 'Texas', 'Tx',
    'FL', 'Florida', 'fl',
    'IL', 'Illinois', 'Ill.',
    'OH', 'Ohio',
    'WA', 'Washington',
]

print('✔  Schema constants defined.')
print(f'   Weather categories  : {weather_map}')
print(f'   Road conditions     : {road_map}')
print(f'   Vehicle types       : {vehicle_map}')

---
## 🎯 Cell 3 — Balanced Target & Severity-Correlated Features
> **WHY balanced classes?** Equal representation of Minor / Moderate / Severe prevents  
> the model from simply predicting the majority class.  
>
> **WHY correlated numerics?** Each numerical feature is generated with a base value  
> that shifts with severity, then Gaussian noise is added to make the signal realistic  
> (i.e. not perfectly separable).

In [ ]:
# ── Balanced target ───────────────────────────────────────────────────────
severity_raw = np.repeat([0, 1, 2], N // 3)
severity_raw = np.concatenate([severity_raw,
                                np.random.choice([0, 1, 2], N - len(severity_raw))])
np.random.shuffle(severity_raw)

g = lambda size, std=5: np.random.normal(0, std, size)   # Gaussian noise helper

# ── Speed (km/h) — higher speed → more severe ─────────────────────────────
base_speed = {0: 45, 1: 70, 2: 105}
speed = np.array([base_speed[s] for s in severity_raw], dtype=float) + g(N, 12)
speed = np.clip(speed, 10, 200)

# ── Vehicles involved — Poisson, mean increases with severity ─────────────
num_vehicles = np.random.poisson(
    [1.2 if s==0 else 2.0 if s==1 else 3.2 for s in severity_raw]
)
num_vehicles = np.clip(num_vehicles, 1, 10)

# ── Driver demographics ───────────────────────────────────────────────────
driver_age = np.clip(np.random.normal(38, 12, N).astype(int), 16, 80)
driver_exp = np.clip(np.random.normal(10, 6, N) + g(N, 2), 0, 50)

# ── Casualties — strong severity correlation ──────────────────────────────
casualties = np.clip(
    np.random.poisson([0.3 if s==0 else 1.2 if s==1 else 3.5 for s in severity_raw]),
    0, 15
)

# ── Alcohol: higher probability for severe accidents ──────────────────────
alcohol_involved = np.array(
    [np.random.binomial(1, [0.05, 0.15, 0.35][s]) for s in severity_raw]
)

# ── Seatbelt: inversely correlated with severity ──────────────────────────
seatbelt_worn = np.array(
    [np.random.binomial(1, [0.92, 0.72, 0.45][s]) for s in severity_raw]
)

# ── Speed limit ───────────────────────────────────────────────────────────
speed_limit = np.random.choice(
    [30, 50, 60, 80, 100, 120], N,
    p=[0.15, 0.25, 0.20, 0.20, 0.12, 0.08]
)

# ── Visibility (m) — lower → worse conditions → higher severity ───────────
base_vis = {0: 800, 1: 500, 2: 250}
visibility = np.array([base_vis[s] for s in severity_raw], dtype=float) + g(N, 80)
visibility = np.clip(visibility, 50, 1200)

print('✔  Severity-correlated numerical features generated.')
print(f'   Speed range       : {speed.min():.1f} – {speed.max():.1f} km/h')
print(f'   Visibility range  : {visibility.min():.0f} – {visibility.max():.0f} m')
print(f'   Casualties range  : {casualties.min()} – {casualties.max()}')

---
## 📅 Cell 4 — Temporal & Categorical Features
> Timestamps span 4 years (2020–2023). Hour-of-day probabilities follow a  
> realistic traffic pattern (peaks at morning and evening rush hours).

In [ ]:
# ── Hour of day — weighted toward peak traffic hours ─────────────────────
hour_probs = np.array([
    0.02, 0.02, 0.02, 0.02, 0.02, 0.02,   # 00–05  (night)
    0.04, 0.06, 0.07, 0.06, 0.05, 0.05,   # 06–11  (morning rush)
    0.04, 0.04, 0.05, 0.05, 0.05, 0.05,   # 12–17  (afternoon)
    0.05, 0.06, 0.06, 0.04, 0.03, 0.02,   # 18–23  (evening rush + late)
])
hour_probs /= hour_probs.sum()   # normalise to exactly 1.0
hour = np.random.choice(range(24), N, p=hour_probs)

day_of_week   = np.random.choice(
    ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'], N
)

# ── Categorical features — assigned with realistic proportions ────────────
weather_raw   = np.random.choice(weather_map,   N, p=[0.35,0.25,0.15,0.10,0.10,0.05])
road_cond     = np.random.choice(road_map,      N, p=[0.40,0.30,0.15,0.10,0.05])
vehicle_type  = np.random.choice(vehicle_map,   N)
light_cond    = np.random.choice(light_map,     N, p=[0.40,0.10,0.10,0.25,0.15])
junction_type = np.random.choice(junction_map,  N, p=[0.30,0.20,0.15,0.25,0.10])
road_type     = np.random.choice(road_type_map, N, p=[0.25,0.30,0.20,0.15,0.10])

# ── Inject categorical inconsistencies (~15 % of weather rows) ───────────
# WHY: Simulates data entered by different people / systems with no enforced schema.
weather_col = [
    random.choice(weather_inconsistent[w]) if np.random.rand() < 0.15 else w
    for w in weather_raw
]

# State column — mix of abbreviations, full names, lowercase
state_col = np.random.choice(state_pool, N)

# ── Timestamps (2020-01-01 to 2023-12-31) ────────────────────────────────
base_date  = datetime(2020, 1, 1)
timestamps = [
    base_date + timedelta(days=int(np.random.uniform(0, 1460)), hours=int(h))
    for h in hour
]
dates = [t.strftime('%Y-%m-%d') for t in timestamps]
times = [t.strftime('%H:%M')    for t in timestamps]

print('✔  Temporal & categorical features generated.')
print(f'   Weather variants in data  : {pd.Series(weather_col).nunique()}')
print(f'   State variants in data    : {pd.Series(state_col).nunique()}')

---
## 🔇 Cell 5 — Noise / Redundant Columns
> **WHY:** Real datasets often contain columns that carry zero predictive signal  
> but clutter the feature space. These are used as a feature-selection challenge  
> in Notebook 2 — any good selection method should discard all four.

| Column | Type | Signal |
|--------|------|--------|
| `redundant_record_id` | Integer row counter | ❌ None — duplicates the index |
| `system_flag` | Constant string `'PROCESSED'` | ❌ None — zero variance |
| `random_noise_code` | Random 6-digit integer | ❌ None — pure noise |
| `useless_ratio` | Random float [0, 1] | ❌ None — no relation to severity |

In [ ]:
redundant_id  = np.arange(1, N + 1)                  # row counter
system_flag   = np.full(N, 'PROCESSED')              # constant value
random_code   = np.random.randint(100000, 999999, N) # random integer
useless_ratio = np.random.uniform(0, 1, N).round(4)  # random float

print('✔  Noise columns created.')

---
## 🔧 Cell 6 — Assemble DataFrame
> All arrays are combined into a single Pandas DataFrame with 26 columns.

In [ ]:
df = pd.DataFrame({
    'accident_id'           : [f'ACC{str(i).zfill(5)}' for i in range(1, N+1)],
    'date'                  : dates,
    'time'                  : times,
    'day_of_week'           : day_of_week,
    'hour'                  : hour,
    'state'                 : state_col,
    'road_type'             : road_type,
    'junction_type'         : junction_type,
    'weather_condition'     : weather_col,
    'road_condition'        : road_cond,
    'light_condition'       : light_cond,
    'speed_kmh'             : speed.round(2),
    'speed_limit_kmh'       : speed_limit,
    'visibility_m'          : visibility.round(1),
    'num_vehicles'          : num_vehicles,
    'driver_age'            : driver_age,
    'driver_experience_yrs' : driver_exp.round(1),
    'vehicle_type'          : vehicle_type,
    'alcohol_involved'      : alcohol_involved,
    'seatbelt_worn'         : seatbelt_worn,
    'num_casualties'        : casualties,
    # ── Noise columns ──
    'redundant_record_id'   : redundant_id,
    'system_flag'           : system_flag,
    'random_noise_code'     : random_code,
    'useless_ratio'         : useless_ratio,
    # ── Target ──
    'accident_severity'     : [SEVERITY_LABELS[s] for s in severity_raw],
})

print(f'✔  DataFrame assembled  : {df.shape}')
df.head()

---
## 💉 Cell 7 — Inject Data Quality Issues
> This is the step that makes the dataset a realistic preprocessing challenge.  
> We intentionally corrupt the clean DataFrame before saving.

### 7.1 Inject 10 % Missing Values

In [ ]:
# WHY: 10 % nulls per column forces the preprocessing notebook to decide
#      between dropping rows, median imputation, or mode imputation.
null_cols = [
    'speed_kmh', 'weather_condition', 'road_condition', 'driver_age',
    'driver_experience_yrs', 'visibility_m', 'light_condition',
    'vehicle_type', 'alcohol_involved', 'seatbelt_worn'
]

for col in null_cols:
    mask = np.random.choice([True, False], len(df), p=[0.10, 0.90])
    df.loc[mask, col] = np.nan

total_nulls = df.isnull().sum().sum()
print(f'✔  Nulls injected : {total_nulls} total across {len(null_cols)} columns')
print(df.isnull().sum()[df.isnull().sum() > 0].to_string())

### 7.2 Inject 5 % Duplicate Rows

In [ ]:
# WHY: Duplicates bias training — the model sees the same example multiple times.
#      Preprocessing must detect and remove them before splitting.
n_dups   = int(0.05 * N)   # 400 duplicate rows
dup_rows = df.sample(n=n_dups, random_state=7)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=99).reset_index(drop=True)  # shuffle

print(f'✔  Duplicates injected : {df.duplicated().sum()} rows  ({n_dups} added)')
print(f'   Final shape          : {df.shape}')

---
## ✅ Cell 8 — Dataset Validation & Save
> Final checks confirm the dataset matches the spec before writing to CSV.

In [ ]:
print('='*55)
print('  DATASET VALIDATION REPORT')
print('='*55)
print(f'  Shape              : {df.shape}')
print(f'  Columns            : {df.shape[1]}')
print(f'  Duplicate rows     : {df.duplicated().sum()}')
print(f'  Total null values  : {df.isnull().sum().sum()}')
print(f'\n  Target distribution:')
print(df["accident_severity"].value_counts().to_string())
print(f'\n  Weather variants (inconsistencies visible):')
print(df["weather_condition"].value_counts().head(12).to_string())
print(f'\n  State variants (inconsistencies visible):')
print(df["state"].value_counts().to_string())
print('='*55)

# Save
df.to_csv('traffic_accident_dataset.csv', index=False)
print('\n✔  Saved → traffic_accident_dataset.csv')

---
## 📊 Cell 9 — Quick Visual Summary

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Dataset Quick Summary', fontsize=13, fontweight='bold')

# Target balance
vc = df['accident_severity'].value_counts()
axes[0].bar(vc.index, vc.values,
            color=['#4C9BE8','#F4A261','#E63946'], edgecolor='white', width=0.5)
axes[0].set_title('Class Balance'); axes[0].set_ylabel('Count')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height()):,}',
                     (p.get_x()+p.get_width()/2, p.get_height()+30),
                     ha='center', fontsize=9)

# Speed distribution by severity
for sev, clr in zip(['Minor','Moderate','Severe'],['#4C9BE8','#F4A261','#E63946']):
    subset = df[df['accident_severity']==sev]['speed_kmh'].dropna()
    axes[1].hist(subset, bins=30, alpha=0.55, color=clr, label=sev, edgecolor='none')
axes[1].set_title('Speed Distribution by Severity')
axes[1].set_xlabel('Speed (km/h)'); axes[1].legend()

# Null counts
null_s = df.isnull().sum().sort_values(ascending=False)
null_s = null_s[null_s > 0]
axes[2].barh(null_s.index[::-1], null_s.values[::-1], color='#cccccc', edgecolor='white')
axes[2].set_title('Missing Values per Column')
axes[2].set_xlabel('Count')

plt.tight_layout()
plt.savefig('dataset_summary.png', dpi=120, bbox_inches='tight')
plt.show()
print('\n✔  Summary chart saved → dataset_summary.png')
print('\n→  Next step: open 02_preprocess_traffic_accidents.ipynb')